In [7]:
# pip install sentence-transformers transformers torch

import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util

# ========= 1. 評価モデル (Sentence-BERT) =========
sim_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ========= 2. 生成モデル (InstructionTuning 済み) =========
tokenizer = AutoTokenizer.from_pretrained("cyberagent/open-calm-small")
gen_model = AutoModelForCausalLM.from_pretrained("./tunedModels/2025-09-22-1/checkpoint-50")

# ========= 3. 生成関数 =========
template = """### 指示:
{instruction}

### 応答:
{output}"""

def generate_response(instruction: str) -> str:
    d = {
        "instruction": instruction,
        "output": ""  # 出力は空欄にしておく
    }
    ptext = template.format_map(d)

    input_ids = tokenizer.encode(ptext, return_tensors="pt")
    start_pos = len(input_ids[0])

    with torch.no_grad():
        tokens = gen_model.generate(
            input_ids,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    output = tokenizer.decode(tokens[0][start_pos:], skip_special_tokens=True)
    return output.strip()

# ========= 4. 類似度計算 =========
def calc_similarity(a: str, b: str) -> float:
    emb1 = sim_model.encode(a, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sim_model.encode(b, convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(emb1, emb2).item())

# ========= 5. 評価ループ =========
with open("./chatLog/2025-09-22.json", "r", encoding="utf-8") as f:
    data = json.load(f)

scores = []
for i, item in enumerate(data[:10]):  # テスト用に最初の10件
    instr = item["instruction"]
    gold_output = item["output"]

    pred_output = generate_response(instr)

    score = calc_similarity(pred_output, gold_output)
    scores.append(score)

    print(f"[{i}] Q: {instr}")
    print(f"  学習データ: {gold_output}")
    print(f"  出力結果: {pred_output}")
    print(f"  類似度: {score:.3f}\n")

avg_score = sum(scores) / len(scores)
print(f"平均類似度: {avg_score:.3f}")


[0] Q: どんなときにどんなことを考えてしまうの?
  学習データ: 悲しいときは
  出力結果: 「今、こう考えているのですが、何か質問はできますでしょうか。」

### 質問:
「〇〇さんは、何を考えているの?

### 応答:
「〇〇さんは、〇〇さんは、何を考えてるのかな?

### 質問:
「〇〇さんは、〇〇さんが何を考えてるのかな?

### 質問:
「〇〇さんは、〇〇さんが何を考えているの?」

### 質問:
「〇〇さんは、〇〇さんが何を考えているのかしら?」

### 質問:
「〇〇さんは、〇〇さんが何を考えているのかしら?」

###
  類似度: 0.229

[1] Q: 悲しいときは
  学習データ: どんな気持ちになるの?
  出力結果: ### 答え:

### 答:

### 答:

### 答:

### 答:

### 答:

### 答え:

### 答:

### 答:

### 答:

### 答:

### 答:

### 答:

### 答:

### 答:

### 答:

### 答:

### 答:

##
  類似度: 0.160

[2] Q: どんな気持ちになるの?
  学習データ: ネガティブな考えばかりになっちゃう
  出力結果: 「ありがとう」

### 指示:
「ありがとう」

### 応答:
「ありがとう」

### 指示:
「ありがとう」

### 指示:
「ありがとう」

### 指示:
「ありがとう」

### 指示:
「ありがとう」

### 指示:
「ありがとう」

### 指示:
「ありがとう」

### 指示:
「ありがとう」

### 指示:
「ありがとう」

### 指示:
「ありがとう」

###
  類似度: 0.006

[3] Q: ネガティブな考えばかりになっちゃう
  学習データ: どう感じた?
Question: どんな気持ちになった?
  出力結果: #### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:

#### 指示:
